<a href="https://colab.research.google.com/github/vcasadei/Redes-Neurais-CESAR-School/blob/2026.01-A/2%20-%20Regress%C3%A3o%20Linear/2.5_regressao_linear_otimizador.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 2.5 — Regressão linear com `nn.MSELoss` e `optim.SGD`

## Objetivos

- Trocar a função de custo (loss) escrita à mão por `nn.MSELoss`.
- Trocar a atualização manual dos parâmetros por um otimizador do PyTorch, `optim.SGD`, com `zero_grad()` e `step()`.
- Verificar que o resultado é o mesmo do 2.4 e próximo da solução analítica.

O problema continua o mesmo: prever o comprimento da pétala da *Iris versicolor* a partir do comprimento da sépala, com a reta

$$\hat{y} = w_0 + w_1 x$$

em que $w_0$ é o bias e $w_1$ é o peso. Nos slides: $h_\theta(x) = \theta_0 + \theta_1 x$, com $w_0 = \theta_0$ e $w_1 = \theta_1$.

O que muda em relação ao 2.4:

| Etapa | 2.4 (autograd) | 2.5 (este notebook) |
|---|---|---|
| Modelo | `nn.Linear(2, 1, bias=False)` | igual |
| Função de custo | `torch.mean((y_pred - y_train)**2)` | `criterion = nn.MSELoss()` |
| Gradiente | `loss.backward()` | igual |
| Atualização | `with torch.no_grad(): model.weight -= lr * model.weight.grad` | `optimizer.step()` |
| Zerar o gradiente | `model.weight.grad.zero_()` | `optimizer.zero_grad()` |

![Fluxo do treinamento](https://raw.githubusercontent.com/vcasadei/images/master/RegressaoLinear_Otimizacao.png)

Na figura, o bloco "Custo" passa a ser o `criterion` e o bloco "Gradiente Descendente" passa a ser o `optimizer`. $\eta$ é a taxa de aprendizado (`lr`), $N$ é o número de amostras e $x_0$ é a entrada $x$.

## Importação das bibliotecas

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_iris

import torch
from torch import nn, optim

torch.manual_seed(1234)
np.random.seed(1234)

## Dados

### Leitura

In [ ]:
iris = load_iris()
# classe 1 (versicolor); colunas 0 e 2: comprimento da sépala e comprimento da pétala
data = iris.data[iris.target == 1, ::2]

x_train = data[:, 0:1].astype(np.float32)   # comprimento da sépala
y_train = data[:, 1:2].astype(np.float32)   # comprimento da pétala

n_samples = x_train.shape[0]
print('x_train.shape:', x_train.shape, x_train.dtype)
print('y_train.shape:', y_train.shape, y_train.dtype)

print('x_train[:5]:\n', x_train[:5])
print('y_train[:5]:\n', y_train[:5])

### Normalização (min-max, para o intervalo [0, 1])

In [ ]:
x_train -= x_train.min()
x_train /= x_train.max()
y_train -= y_train.min()
y_train /= y_train.max()

### Coluna de uns e conversão para tensores

`x_train_bias` tem shape (50, 2): a coluna de uns multiplica $w_0$. Os dados são tensores comuns, sem `requires_grad`.

In [ ]:
x_train_bias = np.hstack([np.ones(shape=(n_samples, 1)), x_train])

x_train_bias = torch.from_numpy(x_train_bias).float()
y_train = torch.from_numpy(y_train)
print('x_train_bias:', x_train_bias.shape, ' y_train:', y_train.shape)

## Modelo

A mesma camada do 2.4: `model.weight` tem shape (1, 2) e guarda $[w_0, w_1]$.

In [ ]:
model = nn.Linear(2, 1, bias=False)

### Inicialização dos parâmetros

Como no 2.4, uniforme em $[-0{,}1;\ 0{,}1]$.

In [ ]:
nn.init.uniform_(model.weight, -0.1, 0.1)
model.weight

### Testando o predict da rede

In [ ]:
model(torch.ones((5, 2)))

## Treinamento

### Função de custo e otimizador

- `nn.MSELoss()` cria um objeto que calcula $\frac{1}{M}\sum_{i=1}^{M}(\hat{y}^{(i)} - y^{(i)})^2$, o mesmo MSE do 2.4 (a média é o padrão, `reduction='mean'`). Chama-se `criterion(previsão, alvo)`, nessa ordem.
- `optim.SGD(model.parameters(), lr=lr)` cria o otimizador de gradiente descendente. `model.parameters()` entrega ao otimizador os tensores que ele vai atualizar; aqui, só `model.weight`.
  - `optimizer.zero_grad()` zera o gradiente de todos esses parâmetros;
  - `optimizer.step()` aplica $w \leftarrow w - \text{lr} \cdot \nabla_w J$ usando o `.grad` calculado por `loss.backward()`.

`criterion` e `optimizer` são nomes de variáveis usados por convenção; `nn.MSELoss` e `optim.SGD` são do PyTorch.

In [ ]:
lr = 0.3
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=lr)

Verificação: com os parâmetros iniciais, `criterion` dá o mesmo valor do custo escrito à mão no 2.4.

In [ ]:
with torch.no_grad():
    y_pred = model(x_train_bias)
    print('nn.MSELoss:  ', criterion(y_pred, y_train).item())
    print('torch.mean:  ', torch.mean((y_pred - y_train) ** 2).item())

### Laço de treinamento

Cada época tem os mesmos passos do 2.4: forward → custo → `zero_grad()` → `backward()` → `step()`. A ordem importa: o gradiente é zerado antes do `backward()`, que o recalcula, e o `step()` vem depois, para usar o gradiente novo.

As listas `w0_list` e `w1_list` guardam os parâmetros a cada época (antes da atualização), para o Exercício 2.

In [ ]:
n_epochs = 120
losses = []
w0_list = []
w1_list = []

for epoch in range(n_epochs):
    # forward: previsão
    y_pred = model(x_train_bias)

    w0_list.append(model.weight[0, 0].item())
    w1_list.append(model.weight[0, 1].item())

    # função de custo
    loss = criterion(y_pred, y_train)

    # backward e otimizador
    optimizer.zero_grad()   # zera os gradientes acumulados
    loss.backward()         # autograd: calcula model.weight.grad
    optimizer.step()        # gradiente descendente: w <- w - lr * grad

    losses.append(loss.item())
    if (epoch + 1) % 20 == 0:
        print(f'Epoch[{epoch + 1}/{n_epochs}], loss: {loss.item():.6f}')

In [ ]:
plt.plot(losses)
plt.xlabel('época')
plt.ylabel('função de custo (MSE)')
plt.title('Custo durante o treinamento')
plt.show()

## Avaliação

In [ ]:
with torch.no_grad():
    y_pred = model(x_train_bias)

plt.plot(x_train, y_train.numpy(), 'ro', label='dados')
plt.plot(x_train, y_pred.numpy(), 'kx-', label='reta ajustada')
plt.xlabel('comprimento da sépala (normalizado)')
plt.ylabel('comprimento da pétala (normalizado)')
plt.legend()
plt.show()

### Comparação com a solução analítica

A equação normal $w^* = (X^T X)^{-1} X^T y$ (2.2 e 2.4) dá os parâmetros ótimos; o modelo treinado deve ficar próximo.

In [ ]:
x_bias_t = x_train_bias.t()
w_opt = torch.inverse(x_bias_t.mm(x_train_bias)).mm(x_bias_t).mm(y_train)
print('w_opt (analítico):', w_opt.t())
print('w treinado:       ', model.weight.detach())

## Exercícios

### Exercício 1 — Custo final com `criterion`

Calcule o valor da função de custo (MSE) depois da rede treinada, usando o `criterion` do laço de treinamento, e guarde o resultado em `loss`. Use `torch.no_grad()`, porque aqui só avaliamos o modelo.

In [ ]:
# TODO: escreva sua solução aqui
print(f'MSE final: {loss.item():.6f}')

### Exercício 2 — Trajetória dos parâmetros

Como no Exercício 3 do 2.4, faça um scatter plot com setas mostrando a evolução de $(w_0, w_1)$ durante o treinamento, usando as listas `w0_list` e `w1_list` já preenchidas no laço. Marque o ponto inicial e o ótimo analítico `w_opt`. Compare com o gráfico do 2.4: o caminho tem a mesma forma?

In [ ]:
# TODO: escreva sua solução aqui


**Resposta:**

*Escreva sua resposta aqui.*

## Aprendizados

- `nn.MSELoss()` calcula o mesmo MSE escrito à mão no 2.4; chama-se `criterion(previsão, alvo)`.
- `optim.SGD(model.parameters(), lr=lr)` recebe os parâmetros a atualizar; `zero_grad()` zera os gradientes e `step()` aplica o passo do gradiente descendente.
- A ordem em cada época é forward → custo → `zero_grad()` → `backward()` → `step()`.
- O otimizador não muda o algoritmo: a trajetória e os parâmetros finais são os mesmos do gradiente descendente manual.